In [7]:
from playground.model.registry import UVModel
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def plot_prediction(
    history: torch.Tensor,
    horizon: torch.Tensor,
    pred: torch.Tensor,
):
    """
    Assume the following inputs:

    - `history`: Tensor of shape (batch_size, seq_len, n_series) representing the historical input data.
    - `horizon`: Tensor of shape (batch_size, horizon_len, n_series) representing the true future values.
    - `pred`: Tensor of shape (batch_size, n_quantiles, horizon_len) representing the predicted quantiles for the future values.
    """
    seq_len = history.shape[1]
    horizon_len = horizon.shape[1]
    n_quantiles = pred.shape[1]

    history_x = np.arange(seq_len)
    horizon_x = np.arange(seq_len, seq_len + horizon_len)

    fig, ax = plt.subplots(figsize=(12, 3))

    ax.plot(history_x, history[0, :, 0].cpu(), label="History", color="xkcd:azure")
    ax.plot(horizon_x, horizon[0, :, 0].cpu(), label="Target Future", color="xkcd:grass green")

    # Median quantile as the main forecast line
    median_idx = n_quantiles // 2
    ax.plot(
        horizon_x,
        pred[0, median_idx, :].cpu().detach(),
        label="Forecast (median)",
        color="xkcd:violet",
    )

    # Outer quantiles as shaded interval
    ax.fill_between(
        horizon_x,
        pred[0, 0, :].cpu().detach(),
        pred[0, -1, :].cpu().detach(),
        alpha=0.4,
        label=f"Prediction interval (q0-q{n_quantiles - 1})",
        color="xkcd:light lavender",
    )

    ax.axvline(x=seq_len - 0.5, color="black", linestyle="--", alpha=0.5, label="Forecast start")
    ax.legend(loc="upper left")
    fig.tight_layout()
    fig.show()

In [6]:
# init model
model = UVModel(
    d_model=256,
    d_ff=512,
    d_kv=32,
    n_heads=8,
    dropout=0.1,
    activation_fn="gelu",
    n_quantiles=9,
    n_encoder_layers=6,
    pred_length=24,
    use_arcsinh=True,
    use_rope=True,
    context_length=512,
    patch_size=16,
    patch_stride=16,
)

# construct dummy input and target data
t = torch.sin(torch.arange(50)).unsqueeze(0).unsqueeze(-1)
horizon = t[:, -24:, :]
t = t[:, :26, :]

print(f"Shape of historical time series: {t.shape}, relates to batch_size, seq_len, n_seq)")
print(f"Shape of forecast horizon: {horizon.shape}, relates to batch_size, horizon_length, n_seq)")

# generate predictions
pred = model(t, n_horizon=24, true_horizon=horizon)

print("Model outputs a tuple of loss and quantile predictions.")
print(f"Loss: {pred[0]}")
print(f"Quantile predictions shape: {pred[1].shape}, related to batch_size, n_quantiles, n_steps(horizion length)")

# visualize the predictions
plot_prediction(t, horizon, pred[1])

Shape of historical time series: torch.Size([1, 26, 1]), relates to batch_size, seq_len, n_seq)
Shape of forecast horizon: torch.Size([1, 24, 1]), relates to batch_size, horizon_length, n_seq)
Model outputs a tuple of loss and quantile predictions.
Loss: 0.3840070962905884
Quantile predictions shape: torch.Size([1, 9, 24]), related to batch_size, n_quantiles, n_steps(horizion length)


NameError: name 'np' is not defined